In [ ]:
# Basic dataset statistics
total_orders = orders.shape[0]
total_users = orders['user_id'].nunique()
total_items = order_products.shape[0]  # total rows in order_products (number of item entries in orders)
unique_products = products['product_id'].nunique()
unique_aisles = products['aisle_id'].nunique()
unique_departments = products['department_id'].nunique()

print(f"Total orders (excluding test): {total_orders}")
print(f"Total users: {total_users}")
print(f"Total ordered item entries: {total_items}")
print(f"Number of unique products: {unique_products}")
print(f"Number of aisles: {unique_aisles}")
print(f"Number of departments: {unique_departments}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- STEP 1: RELOAD THE DATAFRAME ---
# We force reload 'products' to ensure it is a DataFrame, not a list.
# Using r'' handles the backslashes in your Windows path safely.
products = pd.read_csv(r'E:\Dissertation Sem\Papers for dissertation\products.csv')

# --- STEP 2: COMPUTE FREQUENCY ---
# Calculate order frequency
product_counts = order_products['product_id'].value_counts().reset_index()
product_counts.columns = ['product_id', 'order_count']

# --- STEP 3: MERGE (This will now work) ---
product_counts = product_counts.merge(products[['product_id', 'product_name']], on='product_id', how='left')

# Get top 10
top10_products = product_counts.nlargest(10, 'order_count')

# --- STEP 4: PLOT ---
plt.figure(figsize=(10, 6))
sns.barplot(data=top10_products, x='order_count', y='product_name', palette='viridis')
plt.title('Top 10 Most Frequently Ordered Products')
plt.xlabel('Number of Times Ordered')
plt.ylabel('Product Name')
plt.tight_layout()
plt.show()

# Print text results
print("Top 10 products by number of orders:")
for _, row in top10_products.iterrows():
    print(f"{str(row['product_name'])[:30]:30} -- ordered {row['order_count']} times")

In [ ]:
# Calculate basket size for each order (number of items per order)
order_sizes = order_products.groupby('order_id').size()  # pandas Series: index=order_id, value= size

# Plot distribution of basket sizes
plt.figure(figsize=(8, 5))
plt.hist(order_sizes, bins=range(1, 51), color='orange', edgecolor='k')  # bins 1 to 50 for readability
plt.title('Distribution of Basket Sizes (Items per Order)')
plt.xlabel('Number of Items in an Order')
plt.ylabel('Number of Orders')
plt.xlim(0,50)  # focus on 0-50 range for clarity
plt.tight_layout()
plt.show()

# Compute and print some statistics on basket sizes
mean_size = order_sizes.mean()
median_size = order_sizes.median()
mode_size = order_sizes.mode()[0]
print(f"Average basket size: {mean_size:.2f} items")
print(f"Median basket size: {median_size} items")
print(f"Most common basket size (mode): {mode_size} items")


In [ ]:
# Overall reorder rate (fraction of items that are reorders)
overall_reorder_rate = order_products['reordered'].mean()
print(f"Overall, about {overall_reorder_rate*100:.1f}% of ordered items are reorders (previously purchased).")

# Reorder rate per product (we will compute and then examine top/bottom examples)
product_reorder_rate = order_products.groupby('product_id')['reordered'].mean()

# Attach product names for interpretability
product_reorder_rate = product_reorder_rate.reset_index().merge(products[['product_id','product_name']], on='product_id')
product_reorder_rate.rename(columns={'reordered':'reorder_rate'}, inplace=True)

# Get the top 5 products with highest reorder rate (min 100 orders to avoid noise)
min_orders = 100
product_counts = order_products['product_id'].value_counts().reset_index()
product_counts.columns = ['product_id', 'count']
prod_reorder_merged = product_reorder_rate.merge(product_counts, on='product_id')
top5_reordered = prod_reorder_merged[prod_reorder_merged['count']>=min_orders].nlargest(5, 'reorder_rate')
print("\nTop 5 products by reorder rate (among products with 100+ orders):")
for _, row in top5_reordered.iterrows():
    print(f"{row['product_name'][:30]:30}  -  reorder rate: {row['reorder_rate']*100:.1f}%")

# Reorder rate by department
# Map each product_id to its department
order_products['department_id'] = order_products['product_id'].map(products.set_index('product_id')['department_id'])
dept_reorder = order_products.groupby('department_id')['reordered'].mean().reset_index()
dept_reorder = dept_reorder.merge(departments, on='department_id')
dept_reorder.rename(columns={'reordered':'reorder_rate','department':'department_name'}, inplace=True)

print("\nReorder rates by department:")
for _, row in dept_reorder.iterrows():
    print(f"{row['department_name']:20}  {row['reorder_rate']*100:.1f}%")

# Plot: Reorder rate by department
plt.figure(figsize=(8,5))
sns.barplot(data=dept_reorder, x='reorder_rate', y='department_name', color='purple')
plt.title('Average Reorder Rate by Department')
plt.xlabel('Reorder Rate (fraction of items reordered)')
plt.ylabel('Department')
plt.xlim(0, 1)  # 0% to 100%
plt.tight_layout()
plt.show()


In [ ]:
# Map day-of-week numbers to names for clarity
dow_map = {0:'Sunday', 1:'Monday', 2:'Tuesday', 3:'Wednesday', 
           4:'Thursday', 5:'Friday', 6:'Saturday'}
orders['order_dow_name'] = orders['order_dow'].map(dow_map)

# Orders by day of week
orders_by_day = orders['order_dow_name'].value_counts().reindex(
    ['Sunday','Monday','Tuesday','Wednesday','Thursday','Friday','Saturday'])
print("Orders per day of week:")
print(orders_by_day)

plt.figure(figsize=(8,5))
sns.barplot(x=orders_by_day.index, y=orders_by_day.values, palette='Blues_d')
plt.title('Distribution of Orders by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Number of Orders')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Orders by hour of day
orders_by_hour = orders['order_hour_of_day'].value_counts().sort_index()
print("\nOrders per hour of day:")
print(orders_by_hour.head(5), " ...")  # print first few

plt.figure(figsize=(8,5))
sns.barplot(x=orders_by_hour.index, y=orders_by_hour.values, color='green')
plt.title('Distribution of Orders by Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Orders')
plt.xticks(range(0,24,2))  # label every 2 hours for readability
plt.tight_layout()
plt.show()


In [ ]:
# Distribution of days since prior order
intervals = orders['days_since_prior_order'].dropna()  # drop NaN for first orders
plt.figure(figsize=(8,5))
plt.hist(intervals, bins=30, color='steelblue', edgecolor='k')
plt.title('Distribution of Days Between Orders')
plt.xlabel('Days Since Prior Order')
plt.ylabel('Number of Orders')
plt.xticks(range(0, 31, 5))
plt.tight_layout()
plt.show()

# Calculate some stats on the intervals
mean_interval = intervals.mean()
median_interval = intervals.median()
print(f"Average days between orders: {mean_interval:.1f}")
print(f"Median days between orders: {median_interval}")
